# 02 — Index-Pair Relationships

**Purpose:** Establish whether MES–MNQ, MES–M2K, MES–MYM have stable, economically coherent relationships worth hedging (mandate §4.1).

**Research questions:**
1. Are minute returns strongly correlated, and is correlation stable across years and vol regimes?
2. Is the hedged price residual stationary / mean-reverting?
3. Is there intraday seasonality in the relationship?
4. How did the relationships behave through major equity stress (Mar-2020, 2022)?

**Data used:** minute continuous-adjusted closes for MES/MNQ/M2K/MYM (**blocked**, L-001); parent E-mini proxies (ES/NQ/RTY/YM) for pre-2019 context — clearly labeled, never merged.


In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


## Methodology

Per pair: log-price and return scatter; `structural_breaks.rolling_correlation` (multi-window); rolling beta; volatility ratio; residual construction via notebook-04 hedge candidates; `stationarity.stationarity_verdict` + `rolling_adf_pvalue` per year; intraday hour-of-day residual-volatility profile; stress-window event studies. Evidence tags mandatory; per-year stability tables exported to `reports/tables/`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


In [ ]:
if DATA_AVAILABLE:
    from spread_research.data_loader import load_local
    from spread_research.pair_builder import align_pair, build_residual
    from spread_research.hedge_ratios import rolling_ols_beta
    from spread_research.stationarity import stationarity_verdict, rolling_adf_pvalue
    from spread_research.structural_breaks import rolling_correlation
    cfg = yaml.safe_load(open("../config/pair_definitions.yaml"))
    for name, p in cfg["index_pairs"].items():
        a = load_local(p["long_ref"], "minute", DATA_DIR)["close"]
        b = load_local(p["short_ref"], "minute", DATA_DIR)["close"]
        pair = align_pair(a, b)
        ra, rb = np.log(pair["a"]).diff(), np.log(pair["b"]).diff()
        print(name, "| n =", len(pair),
              "| corr(1d roll) median =", float(rolling_correlation(ra, rb, 390).median()))

## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data.

## Limitations

Micro history starts 2019-06 → one broad monetary era plus COVID; parent-proxy analysis extends context but not execution realism (L-005).

## Decision

No pair is accepted or rejected here; verdicts feed notebook 13's aggregation.

## What this means for the algorithm

Pairs failing correlation stability or residual stationarity get dropped before any signal work — this notebook is the first gate in the funnel.